# Notebook de test des modules 

## Imports et configuration

In [1]:
### IMPORTS GENERAUX ###
import os
import sys
import time
import glob
from tqdm.notebook import tqdm
import sacrebleu
import torch
import torch.nn as nn
from torch import optim
import matplotlib.pyplot as plt

BASE_DIR = os.getcwd()

### IMPORTS MODULES ###
sys.path.append(os.path.abspath(os.path.join(BASE_DIR, 'src')))
from data_loader import load_mtedx_data
from asr_pipeline import AudioTranscriber
from lstm_baseline import Lang, EncoderRNN, DecoderRNN, tensorFromSentence, train_epoch, evaluate_lstm
from nmt_pipeline import SubtitleTranslator, MultilingualTranslator
from metrics import evaluate_asr, evaluate_nmt

### CONF PATHS ###


DATA_DIR = os.path.join(os.path.abspath(BASE_DIR), 'data')
OUTPUT_DIR = os.path.join(os.path.abspath(BASE_DIR), 'output')

# Chemins spécifiques datasets test FR
FR_TEST_DIR = os.path.join(os.path.abspath(DATA_DIR), 'fr-fr', 'data', 'test')
FR_WAV_TEST_DIR = os.path.join(FR_TEST_DIR, 'wav')
FR_TXT_TEST_DIR = os.path.join(FR_TEST_DIR, 'txt')
FR_VTT_TEST_DIR = os.path.join(FR_TEST_DIR, 'vtt')

# Chemins spécifiques datasets test EN
EN_TEST_DIR = os.path.join(os.path.abspath(DATA_DIR), 'fr-en', 'data', 'test')
EN_WAV_TEST_DIR = os.path.join(EN_TEST_DIR, 'wav')
EN_TXT_TEST_DIR = os.path.join(EN_TEST_DIR, 'txt')
EN_VTT_TEST_DIR = os.path.join(EN_TEST_DIR, 'vtt')

# Chemins spécifiques datasets test ES
ES_TEST_DIR = os.path.join(os.path.abspath(DATA_DIR), 'fr-es', 'data', 'test')
ES_WAV_TEST_DIR = os.path.join(ES_TEST_DIR, 'wav')
ES_TXT_TEST_DIR = os.path.join(ES_TEST_DIR, 'txt')
ES_VTT_TEST_DIR = os.path.join(ES_TEST_DIR, 'vtt')

# Chemins spécifiques datasets test PT
PT_TEST_DIR = os.path.join(os.path.abspath(DATA_DIR), 'fr-pt', 'data', 'test')
PT_WAV_TEST_DIR = os.path.join(PT_TEST_DIR, 'wav')
PT_TXT_TEST_DIR = os.path.join(PT_TEST_DIR, 'txt')
PT_VTT_TEST_DIR = os.path.join(PT_TEST_DIR, 'vtt')


## Chargement des données

In [2]:
print(f"Chargement des données depuis {DATA_DIR}...")

datasets = load_mtedx_data(DATA_DIR, pairs=['fr-en','fr-es'])

print("\n---- Résultats du chargement ----")
print("Paires de langues chargées :", datasets.keys())

if 'fr-en' in datasets:
    print("\nSplits disponibles pour fr-en :", datasets['fr-en'].keys())
    print(f"Nombre de segments audio (train): {len(datasets['fr-en']['train'])}")

    print("\nExemple de segment audio (segment 0) :")
    exemple = datasets['fr-en']['train'][0]
    for key, value in exemple.items():
        print(f"{key}: {value[:100]}...")

Chargement des données depuis /home/dylan/COURS/M2_COURS/deep_learning/projet_deeplearning/data...
fr-en - train chargé : 30171 phrases.
fr-en - valid chargé : 1036 phrases.
fr-en - test chargé : 1059 phrases.
fr-es - train chargé : 20826 phrases.
fr-es - valid chargé : 1036 phrases.
fr-es - test chargé : 1059 phrases.

---- Résultats du chargement ----
Paires de langues chargées : dict_keys(['fr-en', 'fr-es'])

Splits disponibles pour fr-en : dict_keys(['train', 'valid', 'test'])
Nombre de segments audio (train): 30171

Exemple de segment audio (segment 0) :
src_text: Je m'appelle Julien Le Breton et, comme mon nom ne l'indique pas, je suis né et j'habite en Nouvelle...
tgt_text: I'm Julien Le Breton and as my name doesn't suggest, I was born and live in New Caledonia....


## Transcription sur un fichier audio et evaluation de l'ASR

In [ ]:
print("=== ÉVALUATION ASR (Français) ===")

segments_file = os.path.join(FR_TXT_TEST_DIR, "segments")
text_fr_file = os.path.join(FR_TXT_TEST_DIR, "test.fr")

references_par_audio = {}
if os.path.exists(segments_file) and os.path.exists(text_fr_file):
    with open(segments_file, 'r', encoding='utf-8') as f_seg, open(text_fr_file, 'r', encoding='utf-8') as f_txt:
        segments_lines = f_seg.readlines()
        text_lines = f_txt.readlines()
        for seg_line, txt_line in zip(segments_lines, text_lines):
            parts = seg_line.strip().split()
            if len(parts) >= 2:
                recording_id = parts[1] # ex: '0u7tTptBo9I'
                texte = txt_line.strip()
                if recording_id not in references_par_audio:
                    references_par_audio[recording_id] = []
                references_par_audio[recording_id].append(texte)

for rec_id in references_par_audio:
    references_par_audio[rec_id] = " ".join(references_par_audio[rec_id])

transcripteur = AudioTranscriber(model_name="openai/whisper-small")
hypotheses_asr = []
references_asr = []

fichiers_a_evaluer = list(references_par_audio.keys())[:2]

for rec_id in tqdm(fichiers_a_evaluer, desc="Transcription ASR"):
    audio_path = os.path.join(FR_WAV_TEST_DIR, f"{rec_id}.flac") 
    
    if os.path.exists(audio_path):
        result = transcripteur.pipe(audio_path, generate_kwargs={"language": "french", "task": "transcribe"})
        hypotheses_asr.append(result["text"])
        references_asr.append(references_par_audio[rec_id])
    else:
        print(f"Audio introuvable : {audio_path}")

if hypotheses_asr:
    scores_asr = evaluate_asr(references_asr, hypotheses_asr)
    print(f"Résultats ASR : WER = {scores_asr['WER']}%, CER = {scores_asr['CER']}%\n")
    

=== ÉVALUATION ASR (Français) ===
Chargement du modèle ASR 'openai/whisper-small' sur : cuda:0...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'return_timestamps'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


Transcription ASR:   0%|          | 0/2 [00:00<?, ?it/s]

A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
Whisper did not predict an endi

Résultats ASR : WER = 21.14%, CER = 17.42%



## Approche 1 : LSTM entrainé sur des données de parole et evaluation 

#### Entrainement du modèle LSTM

In [ ]:
# print("=== ENTRAÎNEMENT COMPLET DE LA BASELINE LSTM ===")

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Utilisation de l'appareil : {device}")

# # Chargement des données (Français -> Anglais)
# datasets = load_mtedx_data(DATA_DIR, pairs=['fr-en'])

# # On passe à un gros entraînement : 30 000 phrases !
# # (Tu peux mettre len(datasets['fr-en']['train']) pour utiliser tout le dataset)
# # TRAIN_SIZE = 30000 
# TRAIN_SIZE = len(datasets['fr-en']['train'])
# # Sécurité si le dataset est un peu plus petit
# # TRAIN_SIZE = min(TRAIN_SIZE, len(datasets['fr-en']['train']))
# train_data = datasets['fr-en']['train'].select(range(TRAIN_SIZE))

# print(f"Taille du jeu d'entraînement : {TRAIN_SIZE} phrases.")

# # Construction du vocabulaire
# print("Construction des vocabulaires...")
# input_lang = Lang("fr")
# output_lang = Lang("en")

# for item in train_data:
#     input_lang.add_sentence(item['src_text'].lower())
#     output_lang.add_sentence(item['tgt_text'].lower())

# print(f"Mots uniques FR : {input_lang.n_words}")
# print(f"Mots uniques EN : {output_lang.n_words}")

# # Initialisation du modèle LSTM
# hidden_size = 256
# encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
# decoder = DecoderRNN(hidden_size, output_lang.n_words).to(device)

# learning_rate = 0.01
# encoder_optimizer = optim.SGD(encoder.parameters(), lr=learning_rate)
# decoder_optimizer = optim.SGD(decoder.parameters(), lr=learning_rate)
# criterion = nn.CrossEntropyLoss(ignore_index=2)

# # Boucle d'entraînement aec suivi de la Loss
# epochs = 10
# loss_history = []

# print("\nDébut de l'entraînement massif...")
# for epoch in range(1, epochs + 1):
#     total_loss = 0
#     for item in tqdm(train_data, desc=f"Epoch {epoch}/{epochs}"):
#         input_tensor = tensorFromSentence(input_lang, item['src_text'].lower(), device)
#         target_tensor = tensorFromSentence(output_lang, item['tgt_text'].lower(), device)
        
#         loss = train_epoch(input_tensor, target_tensor, encoder, decoder, 
#                            encoder_optimizer, decoder_optimizer, criterion, device)
#         total_loss += loss
        
#     avg_loss = total_loss / len(train_data)
#     loss_history.append(avg_loss)
#     print(f"Epoch {epoch} terminée | Loss moyenne : {avg_loss:.4f}")

# # --- SAUVEGARDE DU MODÈLE ---
# MODELS_DIR = os.path.join(OUTPUT_DIR, "models")
# os.makedirs(MODELS_DIR, exist_ok=True)
# torch.save(encoder.state_dict(), os.path.join(MODELS_DIR, "encoder_lstm.pt"))
# torch.save(decoder.state_dict(), os.path.join(MODELS_DIR, "decoder_lstm.pt"))
# print(f"\nModèle sauvegardé dans {MODELS_DIR}")

# # --- GRAPHIQUE DE LA LOSS ---
# plt.figure(figsize=(10, 5))
# plt.plot(range(1, epochs + 1), loss_history, marker='o', color='b', label='Training Loss')
# plt.title("Courbe d'apprentissage du modèle LSTM (Baseline)")
# plt.xlabel("Epochs")
# plt.ylabel("Loss (Erreur)")
# plt.grid(True)
# plt.legend()
# plt.savefig(os.path.join(OUTPUT_DIR, "lstm_loss_curve.png"))
# plt.show()

# # ÉVALUATION FINALE (Calcul du BLEU)
# print("\n=== Évaluation du modèle LSTM (BLEU Score) ===")
# # On évalue sur 200 phrases du test set pour comparer avec MarianMT
# test_data = datasets['fr-en']['test']
# EVAL_SIZE = 200
# phrases_test_fr = [item['src_text'].lower() for item in test_data][:EVAL_SIZE]
# references_en = [item['tgt_text'].lower() for item in test_data][:EVAL_SIZE]

# hypotheses_lstm = []
# for phrase in tqdm(phrases_test_fr, desc="Traduction du Test Set"):
#     traduction = evaluate_lstm(encoder, decoder, phrase, input_lang, output_lang, device)
#     hypotheses_lstm.append(traduction)

# # Calcul du BLEU
# bleu_lstm = sacrebleu.corpus_bleu(hypotheses_lstm, [references_en]).score
# print(f"\nSCORE BLEU DU LSTM : {bleu_lstm:.2f}")

# print("\nExemple de traduction LSTM vs Référence :")
# print(f"FR  : {phrases_test_fr[0]}")
# print(f"Vrai: {references_en[0]}")
# print(f"LSTM: {hypotheses_lstm[0]}")

In [ ]:
print("=== CHARGEMENT ET ÉVALUATION DE LA BASELINE LSTM ===")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

datasets = load_mtedx_data(DATA_DIR, pairs=['fr-en'])
TRAIN_SIZE = len(datasets['fr-en']['train'])
train_data = datasets['fr-en']['train'].select(range(TRAIN_SIZE))

input_lang = Lang("fr")
output_lang = Lang("en")
for item in train_data:
    input_lang.add_sentence(item['src_text'].lower())
    output_lang.add_sentence(item['tgt_text'].lower())

hidden_size = 256
encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = DecoderRNN(hidden_size, output_lang.n_words).to(device)

encoder_path = os.path.join(OUTPUT_DIR, "models", "encoder_lstm.pt")
decoder_path = os.path.join(OUTPUT_DIR, "models", "decoder_lstm.pt")

if os.path.exists(encoder_path) and os.path.exists(decoder_path):
    encoder.load_state_dict(torch.load(encoder_path, map_location=device))
    decoder.load_state_dict(torch.load(decoder_path, map_location=device))
    encoder.eval()
    decoder.eval()
    print("Modèles LSTM chargés avec succès depuis ./outputs/models/")
else:
    print("Fichiers .pt introuvables. Le modèle n'est pas chargé.")

# 4. Évaluer sur le jeu de test
test_data = datasets['fr-en']['test']
EVAL_SIZE = 200
phrases_test_fr = [item['src_text'].lower() for item in test_data][:EVAL_SIZE]
references_en = [item['tgt_text'].lower() for item in test_data][:EVAL_SIZE]

hypotheses_lstm = []
for phrase in tqdm(phrases_test_fr, desc="Inférence LSTM"):
    traduction = evaluate_lstm(encoder, decoder, phrase, input_lang, output_lang, device)
    hypotheses_lstm.append(traduction)

bleu_lstm = sacrebleu.corpus_bleu(hypotheses_lstm, [references_en]).score
print(f"\nSCORE BLEU DU LSTM PRÉ-ENTRAÎNÉ : {bleu_lstm:.2f}")

=== CHARGEMENT ET ÉVALUATION DE LA BASELINE LSTM ===
fr-en - train chargé : 30171 phrases.
fr-en - valid chargé : 1036 phrases.
fr-en - test chargé : 1059 phrases.
Modèles LSTM chargés avec succès depuis ./outputs/models/


Inférence LSTM:   0%|          | 0/200 [00:00<?, ?it/s]


SCORE BLEU DU LSTM PRÉ-ENTRAÎNÉ : 1.77


![alt text](output/lstm_loss_curve.png)

## Approche 2 : MNT et Evaluation du NMT

In [ ]:
print("=== ÉVALUATION NMT ===")

nmt_src_file = os.path.join(EN_TXT_TEST_DIR, "test.fr")
nmt_tgt_file_en = os.path.join(EN_TXT_TEST_DIR, "test.en")
nmt_tgt_file_es = os.path.join(ES_TXT_TEST_DIR, "test.es")
targets_disponibles = [nmt_tgt_file_en, nmt_tgt_file_es]

ext_model = {
    "en": "Helsinki-NLP/opus-mt-fr-en",
    "es": "Helsinki-NLP/opus-mt-fr-es"
}

for nmt_tgt_file in targets_disponibles:
    extension = os.path.basename(nmt_tgt_file).split(".")[-1]
    print(f"\n--- Évaluation NMT pour la langue cible : {extension.upper()} ---")
    if os.path.exists(nmt_src_file) and os.path.exists(nmt_tgt_file):
        with open(nmt_src_file, 'r', encoding='utf-8') as f_src, open(nmt_tgt_file, 'r', encoding='utf-8') as f_tgt:
            phrases_fr = [line.strip() for line in f_src.readlines()]
            references = [line.strip() for line in f_tgt.readlines()]
        EVAL_SIZE = 100
        phrases_fr = phrases_fr[:EVAL_SIZE]
        references = references[:EVAL_SIZE]
        
        model_name = ext_model.get(extension)
        if not model_name:
            print(f"Pas de modèle configuré pour l'extension {extension}")
            continue
        
        traducteur = SubtitleTranslator(model_name=model_name)
        
        hypotheses = []
        for phrase in tqdm(phrases_fr, desc=f"Traduction NMT {extension}"):
            hypotheses.append(traducteur.translate_text(phrase))
            
        scores_nmt = evaluate_nmt(references, hypotheses)
        print(f"Résultats NMT ({extension.upper()}) : BLEU = {scores_nmt['BLEU']}, chrF = {scores_nmt['chrF']}")
    else:
        print(f"Fichiers de traduction introuvables pour {nmt_tgt_file}")

=== ÉVALUATION NMT ===

--- Évaluation NMT pour la langue cible : EN ---
Chargement du modèle NMT 'Helsinki-NLP/opus-mt-fr-en' sur : cuda:0...


/home/dylan/.cache/pypoetry/virtualenvs/projet-deeplearning-aNZGoVd5-py3.11/lib/python3.11/site-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

Traduction NMT en:   0%|          | 0/100 [00:00<?, ?it/s]

Résultats NMT (EN) : BLEU = 42.99, chrF = 64.52

--- Évaluation NMT pour la langue cible : ES ---
Chargement du modèle NMT 'Helsinki-NLP/opus-mt-fr-es' sur : cuda:0...


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Traduction NMT es:   0%|          | 0/100 [00:00<?, ?it/s]

Résultats NMT (ES) : BLEU = 49.77, chrF = 69.28


## Approche 3 : NLLB-200 et evaluation 

In [ ]:
print("=== ÉVALUATION DU MODÈLE NLLB-200 (Anglais, Espagnol, Portugais) ===")
traducteur_multi = MultilingualTranslator(model_name="facebook/nllb-200-distilled-600M")

paires_a_evaluer = ['fr-en', 'fr-es', 'fr-pt']
datasets_multi = load_mtedx_data(DATA_DIR, pairs=paires_a_evaluer)

# Dictionnaire des codes langues pour NLLB
codes_langues = {"fr-en": "eng_Latn", "fr-es": "spa_Latn", "fr-pt": "por_Latn"}

scores_multilingues = {}

for paire in paires_a_evaluer:
    if paire in datasets_multi:
        print(f"\nEvaluation pour la paire : {paire}")
        test_data = datasets_multi[paire]['test']
        
        # On limite à 100 phrases pour que le Notebook s'exécute rapidement
        EVAL_SIZE = 100
        phrases_fr = [item['src_text'] for item in test_data][:EVAL_SIZE]
        references = [item['tgt_text'] for item in test_data][:EVAL_SIZE]
        
        hypotheses = []
        for phrase in tqdm(phrases_fr, desc=f"Traduction {paire}"):
            traduction = traducteur_multi.translate_text(phrase, tgt_lang=codes_langues[paire])
            hypotheses.append(traduction)
   
        bleu_score = sacrebleu.corpus_bleu(hypotheses, [references]).score
        chrf_score = sacrebleu.corpus_chrf(hypotheses, [references]).score
        
        scores_multilingues[paire] = {"BLEU": bleu_score, "chrF": chrf_score}
        print(f"Résultats {paire} (NLLB) : BLEU = {bleu_score:.2f} | chrF = {chrf_score:.2f}")

def generer_srt_depuis_dataset(paire_eval="fr-en", nb_segments=50):
    """
    Utilise les segments officiels du dataset pour créer un SRT parfait.
    """
    langues = {
        "en": "eng_Latn",
        "es": "spa_Latn",
        "pt": "por_Latn",
        "it": "ita_Latn",
        "de": "deu_Latn"
    }

    test_data = datasets_multi['fr-en']['test'].select(range(nb_segments))

    
    for ext, code_nllb in langues.items():
        print(f"Traduction haute qualité (Segments Dataset) vers : {ext.upper()}...")
        
        lignes_srt = []
        for i, item in enumerate(test_data):
            start_time = i * 4
            end_time = (i + 1) * 4
            
            # Formattage SRT : 00:00:00,000
            def to_srt_time(seconds):
                h = int(seconds // 3600)
                m = int((seconds % 3600) // 60)
                s = int(seconds % 60)
                return f"{h:02d}:{m:02d}:{s:02d},000"

            # Traduction du texte propre du dataset
            texte_traduit = traducteur_multi.translate_text(item['src_text'], tgt_lang=code_nllb)
            
            lignes_srt.append(f"{i+1}\n")
            lignes_srt.append(f"{to_srt_time(start_time)} --> {to_srt_time(end_time)}\n")
            lignes_srt.append(f"{texte_traduit}\n\n")
            
        output_path = os.path.join(OUTPUT_DIR, "srt", f"output_{ext}.srt")
        with open(output_path, 'w', encoding='utf-8') as f:
            f.writelines(lignes_srt)
        print(f"Fichier sauvegardé : {output_path}")

generer_srt_depuis_dataset(nb_segments=50)

=== ÉVALUATION DU MODÈLE NLLB-200 (Anglais, Espagnol, Portugais) ===
Chargement du modèle Multilingue 'facebook/nllb-200-distilled-600M' sur : cuda:0...


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


fr-en - train chargé : 30171 phrases.
fr-en - valid chargé : 1036 phrases.
fr-en - test chargé : 1059 phrases.
fr-es - train chargé : 20826 phrases.
fr-es - valid chargé : 1036 phrases.
fr-es - test chargé : 1059 phrases.
fr-pt - train chargé : 13286 phrases.
fr-pt - valid chargé : 1036 phrases.
fr-pt - test chargé : 1059 phrases.

Evaluation pour la paire : fr-en


Traduction fr-en:   0%|          | 0/100 [00:00<?, ?it/s]

Résultats fr-en (NLLB) : BLEU = 39.97 | chrF = 62.64

Evaluation pour la paire : fr-es


Traduction fr-es:   0%|          | 0/100 [00:00<?, ?it/s]

Résultats fr-es (NLLB) : BLEU = 44.17 | chrF = 66.38

Evaluation pour la paire : fr-pt


Traduction fr-pt:   0%|          | 0/100 [00:00<?, ?it/s]

Résultats fr-pt (NLLB) : BLEU = 41.84 | chrF = 66.53
🌍 Traduction haute qualité (Segments Dataset) vers : EN...
✅ Fichier sauvegardé : ./output/srt/perfect_segments_en.srt
🌍 Traduction haute qualité (Segments Dataset) vers : ES...
✅ Fichier sauvegardé : ./output/srt/perfect_segments_es.srt
🌍 Traduction haute qualité (Segments Dataset) vers : PT...
✅ Fichier sauvegardé : ./output/srt/perfect_segments_pt.srt
🌍 Traduction haute qualité (Segments Dataset) vers : IT...
✅ Fichier sauvegardé : ./output/srt/perfect_segments_it.srt
🌍 Traduction haute qualité (Segments Dataset) vers : DE...
✅ Fichier sauvegardé : ./output/srt/perfect_segments_de.srt


## Cascade Errors

In [ ]:
print("=== EXPÉRIENCE : LES ERREURS EN CASCADE (ASR -> NMT) POUR TOUTES LES LANGUES ===")

# fichiers source / segments
segments_file = os.path.join(FR_TXT_TEST_DIR, "segments")
test_fr_file = os.path.join(FR_TXT_TEST_DIR, "test.fr")

lang_configs = {
    "EN": {"txt": os.path.join(EN_TXT_TEST_DIR, "test.en"),
           "model": "Helsinki-NLP/opus-mt-fr-en"},
    "ES": {"txt": os.path.join(ES_TXT_TEST_DIR, "test.es"),
           "model": "Helsinki-NLP/opus-mt-fr-es"}
}

with open(segments_file, 'r', encoding='utf-8') as f:
    segments = f.readlines()

first_rec_id = segments[0].split()[1]
print(f"Analyse sur la vidéo ID : {first_rec_id}")

indices_lignes = [i for i, line in enumerate(segments)
                  if line.split()[1] == first_rec_id]

with open(test_fr_file, 'r', encoding='utf-8') as f:
    lignes_fr = f.readlines()

phrases_gt_fr = [lignes_fr[i].strip() for i in indices_lignes]

audio_path = os.path.join(FR_WAV_TEST_DIR, f"{first_rec_id}.flac")
transcripteur = AudioTranscriber(model_name="openai/whisper-small")

print("\n1/3 - Whisper écoute et transcrit l'audio (ASR)...")
resultat_asr = transcripteur.pipe(audio_path,
                                  generate_kwargs={"language": "french",
                                                   "task": "transcribe"})
chunks_asr_fr = [chunk['text'] for chunk in resultat_asr['chunks']]

for lang_label, config in lang_configs.items():
    tgt_file = config["txt"]
    if not os.path.exists(tgt_file):
        print(f"\nFichier de référence introuvable pour {lang_label}: {tgt_file}")
        continue

    with open(tgt_file, 'r', encoding='utf-8') as f:
        lignes_tgt = f.readlines()

    phrases_gt_tgt = [lignes_tgt[i].strip() for i in indices_lignes]
    texte_ref = " ".join(phrases_gt_tgt)

    traducteur = SubtitleTranslator(model_name=config["model"])
    print(f"\n--- CIBLE {lang_label} ---")
    print("2/3 - Traduction du texte PARFAIT (Ground Truth)...")
    traductions_gt = [traducteur.translate_text(p) for p in phrases_gt_fr]
    texte_gt = " ".join(traductions_gt)

    print("3/3 - Traduction de la sortie WHISPER (Cascade Error)...")
    traductions_asr = [traducteur.translate_text(c) for c in chunks_asr_fr]
    texte_asr = " ".join(traductions_asr)

    bleu_ideal = sacrebleu.sentence_bleu(texte_gt, [texte_ref]).score
    bleu_cascade = sacrebleu.sentence_bleu(texte_asr, [texte_ref]).score

    print(f"BLEU idéal : {round(bleu_ideal, 2)}")
    print(f"BLEU en cascade : {round(bleu_cascade, 2)}")
    print(f"Perte due à l'audio : -{round(bleu_ideal - bleu_cascade, 2)} points BLEU")

    print("\nExtrait qualitatif :")
    print(f"GT FR   : {phrases_gt_fr[0]}")
    print(f"Whisper : {chunks_asr_fr[0].strip()}")
    print(f"Trad GT : {traductions_gt[0]}")
    print(f"Trad ASR: {traductions_asr[0]}")

=== EXPÉRIENCE : LES ERREURS EN CASCADE (ASR -> NMT) POUR TOUTES LES LANGUES ===
Analyse sur la vidéo ID : 0u7tTptBo9I
Chargement du modèle ASR 'openai/whisper-small' sur : cuda:0...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).



1/3 - Whisper écoute et transcrit l'audio (ASR)...


### Métriques Quantitatives

Nous avons évalué nos pipelines sur les sous-ensembles de test de `mTEDx`. Voici le récapitulatif de nos expériences.

**1. Évaluation ASR (Transcription Audio)**
* **Modèle utilisé :** `openai/whisper-small`
* **WER (Word Error Rate) :** 21.14%
* **CER (Character Error Rate) :** 17.42%
* *Analyse :* Le modèle génère environ 1 erreur sur 5 mots. Ces erreurs sont généralement dues aux bruits de fond, aux hésitations de l'orateur ou aux mots hors vocabulaire (noms propres).

**2. Évaluation NMT (Traduction Neuronale)**

| Modèle (Approche) | Paire de langues | VRAM estimée | Score BLEU | Score chrF |
| :--- | :--- | :--- | :--- | :--- |
| **LSTM** (Baseline From Scratch) | FR ➔ EN | ~0.2 Go | **1.77** | N/A |
| **MarianMT** (Transformer Spécialisé) | FR ➔ EN | ~0.5 Go | **42.99** | **64.52** |
| **MarianMT** (Transformer Spécialisé) | FR ➔ ES | ~0.5 Go | **49.77** | **69.28** |
| **NLLB-200** (Massivement Multilingue) | FR ➔ EN | ~2.5 Go | 39.97 | 62.64 |
| **NLLB-200** (Massivement Multilingue) | FR ➔ ES | ~2.5 Go | 44.17 | 66.38 |
| **NLLB-200** (Massivement Multilingue) | FR ➔ PT | ~2.5 Go | **41.84** | **66.53** |

### Analyse des Ressources et du Matériel
* **Matériel utilisé :** NVIDIA GeForce RTX 3060 Laptop GPU (6.0 Go de VRAM dédiée), 16 Go de RAM système, CPU 3200 MT/s.
* **Gestion de la mémoire :** La contrainte majeure de cette pipeline est la **VRAM limitée (6 Go)**. Whisper (~1.5 Go), MarianMT (~0.5 Go) et NLLB-200 (~2.5 Go) ne peuvent pas coexister simultanément dans la VRAM sous peine de déclencher une erreur `CUDA Out of Memory`. Il a été nécessaire d'implémenter des fonctions de nettoyage (`torch.cuda.empty_cache()` et Garbage Collection) entre chaque étape d'inférence.

## Erreurs en cascade (ASR puis NMT)

## Analsyse Qualitative et Limites
**3. Typologie d'erreurs et Limites des modèles :**
* **Contresens & Omissions :** L'approche **LSTM (Baseline)** obtient un score BLEU très faible (1.77) en raison du problème d'évanouissement du gradient ("vanishing gradient") et du goulot d'étranglement de son vecteur de contexte. Il est incapable de conserver le sens sur des phrases de plus de 10 mots et génère le token `<EOS>` prématurément.
* **MarianMT vs NLLB-200 :** Les deux architectures basées sur l'Attention ("Transformers") pulvérisent la baseline récurrente. On remarque que **MarianMT est systématiquement supérieur à NLLB-200** sur ses langues de spécialité (+3.02 points BLEU en Anglais, +5.6 points BLEU en Espagnol). Cependant, NLLB compense cette légère baisse de performance par sa flexibilité : un seul modèle permet de traduire vers le Portugais (41.84 BLEU) et de générer du *Zero-Shot* vers des langues non évaluables ici (Italien, Allemand).

**4. Impact des erreurs ASR sur la traduction (Cascade Error) :**
* *Expérience :* Lors de nos tests, la traduction NMT d'un texte francais parfait (Ground Truth) a donné un BLEU de **45.30**. En traduisant la transcription générée par Whisper pour ce même texte, le score BLEU s'effondre à **27.20** (soit une **perte nette de -18.1 points BLEU**).
* *Explication qualitative :* Cette chute brutale ("Cascade Error") s'explique par les hallucinations de l'ASR (ex: invention de mots-valises comme *"pourtantrangement"*) et la retranscription phonétique des entités nommées (ex: *"Heathcote Williams"* transcrit en *"Edcott Williams"*). Le modèle de traduction, entraîné sur un corpus parfaitement orthographié et ponctué, n'est pas robuste face au bruit généré par le pipeline Speech-to-Text.